# Baseline U-Net Training — Low-Dose CT Denoising (Colab GPU)

This notebook runs the **full** Week 2 pipeline on a free Colab GPU:
generate the synthetic dataset → train the baseline U-Net → plot curves →
show qualitative results → download the `baseline.pt` checkpoint.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

The dev laptop is CPU-only, so training happens here; the checkpoint is then
downloaded and served by the local FastAPI backend (Weeks 6–7).

## 1. Get the code and confirm the GPU

In [ ]:
import os

REPO = 'PINN-CT-Denoising-Web-App'
PATH = f'/content/{REPO}'

# Idempotent: clone on a fresh runtime, otherwise just pull the latest commits.
# Lets you re-run this cell after pushing a fix without restarting the runtime.
if not os.path.exists(PATH):
    !git clone https://github.com/KendoCee25/{REPO}.git {PATH}

%cd {PATH}
!git pull
!git log --oneline -1

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# Colab ships torch + scikit-image; install anything missing quietly.
!pip install -q scikit-image

## 2. Generate the dataset (>=500 phantoms x 3 dose levels)

Runs on the GPU. Bump `--size` to 256 for higher-resolution training if time allows.

In [ ]:
!python -m training.make_dataset --n-phantoms 500 --size 128 --device cuda --out data/dataset

## 3. Train the baseline U-Net (MSE loss)

In [ ]:
!python -m training.train --data-dir data/dataset --epochs 40 --batch-size 32 \
    --lr 1e-3 --device cuda --out models/baseline.pt

## 4. Plot training curves (convergence)

In [ ]:
import json
import matplotlib.pyplot as plt

h = json.load(open('models/baseline.curves.json'))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h['epoch'], h['train_loss']); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(h['epoch'], h['val_psnr']); ax[1].set_title('Val PSNR (dB)'); ax[1].set_xlabel('epoch')
ax[2].plot(h['epoch'], h['val_ssim']); ax[2].set_title('Val SSIM'); ax[2].set_xlabel('epoch')
plt.tight_layout(); plt.show()
print('Best val PSNR:', max(h['val_psnr']), 'dB | Best val SSIM:', max(h['val_ssim']))

## 4b. Per-dose evaluation vs. the do-nothing baseline

**The number that matters.** A denoiser must be compared against passing the noisy
input straight through, not judged on its PSNR alone. A high absolute PSNR can
still be a *regression* if the input was already cleaner than the output.

`delta > 0` at every dose is the pass condition. This table goes in the dissertation.

In [ ]:
import numpy as np
import torch
from training.unet import UNet
from training.metrics import psnr, ssim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('models/baseline.pt', map_location=device)
model = UNet().to(device); model.load_state_dict(ckpt['model_state']); model.eval()

test = np.load('data/dataset/test.npz')
clean, noisy, dose = test['clean'], test['noisy'], test['dose']
DOSE_NAMES = {0: 'low', 1: 'medium', 2: 'high'}

print(f"{'dose':<8} {'input PSNR/SSIM':<22} {'U-Net PSNR/SSIM':<22} {'delta PSNR':<12}")
print('-' * 66)
rows = []
for code in [0, 1, 2, None]:
    idx = np.arange(len(clean)) if code is None else np.where(dose == code)[0]
    in_p, in_s, out_p, out_s = [], [], [], []
    for start in range(0, len(idx), 64):
        batch = idx[start:start + 64]
        x = torch.from_numpy(noisy[batch])[:, None].to(device)
        with torch.no_grad():
            y = model(x).clamp(0, 1).cpu().numpy()[:, 0]
        for k, i in enumerate(batch):
            in_p.append(psnr(noisy[i], clean[i])); in_s.append(ssim(noisy[i], clean[i]))
            out_p.append(psnr(y[k], clean[i]));    out_s.append(ssim(y[k], clean[i]))
    name = 'ALL' if code is None else DOSE_NAMES[code]
    d = np.mean(out_p) - np.mean(in_p)
    rows.append((name, np.mean(in_p), np.mean(in_s), np.mean(out_p), np.mean(out_s), d))
    print(f'{name:<8} {np.mean(in_p):6.2f} dB / {np.mean(in_s):.4f}      '
          f'{np.mean(out_p):6.2f} dB / {np.mean(out_s):.4f}      {d:+.2f} dB')

print()
if all(r[5] > 0 for r in rows):
    print('PASS - the U-Net improves on the noisy input at every dose level.')
else:
    worst = min(rows, key=lambda r: r[5])
    print(f'FAIL - worse than doing nothing at "{worst[0]}" ({worst[5]:+.2f} dB). '
          'Check the noise calibration before training the PINN.')

## 5. Qualitative check: noisy input vs U-Net output vs clean

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from training.unet import UNet
from training.metrics import psnr_ssim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('models/baseline.pt', map_location=device)
model = UNet().to(device); model.load_state_dict(ckpt['model_state']); model.eval()

test = np.load('data/dataset/test.npz')
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for row in range(3):
    i = np.random.randint(len(test['clean']))
    noisy = torch.from_numpy(test['noisy'][i])[None, None].to(device)
    clean = test['clean'][i]
    with torch.no_grad():
        out = model(noisy).clamp(0, 1).cpu().numpy()[0, 0]
    n = test['noisy'][i]
    for ax, img, title in zip(axes[row], [n, out, clean], ['noisy', 'U-Net', 'clean']):
        ax.imshow(img, cmap='gray', vmin=0, vmax=1); ax.axis('off')
        if title != 'clean':
            p, s = psnr_ssim(img, clean)
            ax.set_title(f'{title}\nPSNR {p:.1f} | SSIM {s:.3f}')
        else:
            ax.set_title('clean (ground truth)')
plt.tight_layout(); plt.show()

## 6. Download the checkpoint
Save `baseline.pt` locally into the repo's `models/` folder for the backend.

In [ ]:
from google.colab import files
files.download('models/baseline.pt')
files.download('models/baseline.curves.json')